In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from os import path

/home/greg/Documents/Estudos/CURSO_ASIMOV/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text",
)

In [3]:
arquvios = [
    path.join('/', 'tmp', 'Explorando a API da OpenAI.pdf'),
    path.join('/', 'tmp', 'Explorando o Universo das IAs com Hugging Face.pdf')
]

documentos = []
for arquivo in arquvios:
    pdf = PyPDFLoader(arquivo)
    docs = pdf.load()
    documentos.extend(docs)

In [4]:
chunks_size = 500
chunks_overlap = 50
separator = ["\n\n", "\n", " ", ".", ","]

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunks_size,
    chunk_overlap=chunks_overlap,
    separators=separator
)

##### Criando a base de dados

In [5]:
db_vector = path.join('/', 'tmp', 'db_vector')

vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory=db_vector
)

#### Retrieval
Aqui estamos usando o similarity_search, o que trás muita imprecisão no resultado. Trazendo texto com alta taxa de similaridade

In [6]:
pergunta = "Quais são os principais pontos do livro?"

for resposta in vector_store.similarity_search(pergunta, k=2):
    print(resposta.page_content)
    print(resposta.metadata)
    print('-'*100)

Explorando o Universo das IAs com Hugging Face
Hugging Face, desde que você utilize a configuração mais básica.
Figure 27: Página inicial dos Spaces do Hugging Face
Este link apresenta os principais features dos Spaces do Hugging Face.
Asimov Academy 73
{'trapped': '/False', 'subject': '', 'total_pages': 89, 'page': 73, 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'producer': 'pdfTeX-1.40.25', 'keywords': '', 'title': 'Explorando o Universo das IAs com Hugging Face', 'page_label': '73', 'source': '/tmp/Explorando o Universo das IAs com Hugging Face.pdf', 'moddate': '2024-03-20T18:50:05-03:00', 'creationdate': '2024-03-20T18:50:05-03:00', 'creator': 'LaTeX via pandoc with the Eisvogel template', 'author': 'Asimov Academy'}
----------------------------------------------------------------------------------------------------
Explorando o Universo das IAs com Hugging Face
Figure 34: Erro no aplicativo
O problema aqui é q

In [7]:
pergunta = "O que é OpenAI?"

for resposta in vector_store.similarity_search(pergunta, k=2):
    print(resposta.page_content)
    print(resposta.metadata)    
    print('-'*100)

Explorando o Universo das IAs com
Hugging Face
Asimov Academy
{'creator': 'LaTeX via pandoc with the Eisvogel template', 'author': 'Asimov Academy', 'trapped': '/False', 'subject': '', 'producer': 'pdfTeX-1.40.25', 'source': '/tmp/Explorando o Universo das IAs com Hugging Face.pdf', 'keywords': '', 'creationdate': '2024-03-20T18:50:05-03:00', 'page_label': '1', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'total_pages': 89, 'title': 'Explorando o Universo das IAs com Hugging Face', 'page': 0, 'moddate': '2024-03-20T18:50:05-03:00'}
----------------------------------------------------------------------------------------------------
Explorando o Universo das IAs com Hugging Face
Figure 38: Área de Settings do Space.
Em seguida, vá até o tópicoVariables and secrets , clique em New secret e adicione o valor
do seu token (dentro do seu arquivo local .env). Lembre-se de dar o mesmo nome que é usado pelo
código do webapp:


In [8]:
pergunta = "O que a apostila de OpenAI fala?"

for resposta in vector_store.similarity_search(pergunta, k=2):
    print(resposta.page_content)
    print(resposta.metadata.get('source'))    
    print('-'*100)

Explorando o Universo das IAs com Hugging Face
Figure 32: Salvando o código no seu Space
Passo 3: Adicione as dependências
Seu código está no ar! Vamos checar seu aplicativo na abaApp:
Figure 33: Selecionando a aba App do seu Space.
Contudo, seu aplicativo deve estar com um erro (você pode clicar emLogs no topo da tela para ter a
informação completa do terminal):
Asimov Academy 78
/tmp/Explorando o Universo das IAs com Hugging Face.pdf
----------------------------------------------------------------------------------------------------
Explorando o Universo das IAs com Hugging Face
Figure 34: Erro no aplicativo
O problema aqui é que nunca especificamos as dependências (bibliotecas) que seu webapp precisa.
Para fazer isso, volte para a abaFiles, e clique emAdd file e em seguida Create a new file
(note também que alguns arquivos de configuração foram criados pelo Hugging Face, além do seu
app.py):
Asimov Academy 79
/tmp/Explorando o Universo das IAs com Hugging Face.pdf
------------------

#### Podemos observar alguns problemas acima: 
- Busca por similaridade muito grande.
- Algumas vezes repete os resultados por conta da similaridade.

Mas abaixo vamos contornar

##### MMR
Max marginal relevance

In [9]:
pergunta = "O que é o OpenAI?"

for resposta in vector_store.max_marginal_relevance_search(pergunta, fetch_k=10, k=2):
    print(resposta.page_content)
    print(resposta.metadata.get('source'))
    print('-'*100)

Explorando o Universo das IAs com
Hugging Face
Asimov Academy
/tmp/Explorando o Universo das IAs com Hugging Face.pdf
----------------------------------------------------------------------------------------------------
Explorando o Universo das IAs com Hugging Face
Figure 31: Editando o arquivo app.py.
E em seguida, clicar emCommit new file to main (assegure que a opçãocommit directly
to main esteja selecionada):
Asimov Academy 77
/tmp/Explorando o Universo das IAs com Hugging Face.pdf
----------------------------------------------------------------------------------------------------


##### Filtragem

In [ ]:
pergunta = "O que é o IA?"

for resposta in vector_store.similarity_search(pergunta, k=2, 
                filter={"source": "/tmp/Explorando o Universo das IAs com Hugging Face.pdf"}
                ):
    print(resposta.page_content)
    print(resposta.metadata.get('source'))
    print('-'*100)

Explorando o Universo das IAs com
Hugging Face
Asimov Academy
/tmp/Explorando o Universo das IAs com Hugging Face.pdf
----------------------------------------------------------------------------------------------------
Explorando o Universo das IAs com Hugging Face
Figure 38: Área de Settings do Space.
Em seguida, vá até o tópicoVariables and secrets , clique em New secret e adicione o valor
do seu token (dentro do seu arquivo local .env). Lembre-se de dar o mesmo nome que é usado pelo
código do webapp:
Asimov Academy 83
/tmp/Explorando o Universo das IAs com Hugging Face.pdf
----------------------------------------------------------------------------------------------------


In [15]:
pergunta = "O que é pytorch_model?"

for resposta in vector_store.similarity_search(pergunta, k=2, 
                filter={"page": {"$in": list(range(20, 30))}}
                ):
    print(resposta.page_content)
    print(resposta.metadata.get('source'))
    print('-'*100)

Explorando o Universo das IAs com Hugging Face
Tamanho do arquivo pytorch_model.bin
Na aba de arquivos, há sempre um arquivo chamadopytorch_model.bin. Este arquivo representa
os parâmetros de um modelo treinado. Em geral, uma boa regra é supor que o modelo rodando
precisará de 20% a mais de memória que o seu tamanho indicado.
Asimov Academy 20
/tmp/Explorando o Universo das IAs com Hugging Face.pdf
----------------------------------------------------------------------------------------------------
Explorando o Universo das IAs com Hugging Face
Figure 12: Arquivo pytorch_model.bin do modelo bert-base-portuguese-cased.
Em outras palavras, para o modelo bert-base-portuguese-cased, será necessário cerca de
525 MB de memória RAM!
Asimov Academy 21
/tmp/Explorando o Universo das IAs com Hugging Face.pdf
----------------------------------------------------------------------------------------------------


#### LLM Aided Retrieval


In [18]:
from langchain_community.llms import OpenAI
from langchain_classic.retrievers import SelfQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo

In [ ]:
metadata_info = [
    AttributeInfo(
        name="source",
        description="O nome do arquivo",
        is_filter=True
    )
]